# earthToPixels — tile-stylization exploration

The mask-based infill path is gone. This notebook now does plain image-to-image: take **one** image, send to a model, get back the stylized version.

The input can be:

- a **single tile PNG** (path) — sent as-is
- a **folder** of 9 tiles → automatically stitched into one 3×3 grid image and sent as a whole
- a `PIL.Image` or raw PNG bytes

Naming conventions accepted for folders:

- `0.png`..`8.png` in reading order (top-left → bottom-right)
- `c{col}_r{row}.png` with `col, row ∈ {-1, 0, 1}` — the format produced by `/debug/renderer`'s "render random samples" button

Providers (mask-based-only providers dropped):

- `gpt-image-1`, `gpt-image-1.5`, `gpt-image-2` — OpenAI image-to-image without mask
- Optional `_HIGH` variants pass `input_fidelity='high'`

Why drop the mask? OpenAI's gpt-image treats the mask as a soft hint at best, and the hybrid layout was confusing the model into hallucinating geography. Sending the full image with no mask gets faithful content and the model restyles consistently across the frame.

## 1. Setup

Install deps once (uncomment), then load env from the repo root (same `.env` the TS code reads).

In [ ]:
!pip install python-dotenv openai pillow matplotlib
# Optional (Imagen via Vertex AI):
!pip install google-cloud-aiplatform

In [ ]:
import base64
import io
import os
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Optional, Union

from dotenv import load_dotenv
from PIL import Image


def find_repo_root(start: Path) -> Path:
    """Walk upward until we hit pnpm-workspace.yaml — same heuristic as @mapart/env's loader."""
    cur = start.resolve()
    for _ in range(10):
        if (cur / 'pnpm-workspace.yaml').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError('repo root not found (no pnpm-workspace.yaml walking up)')


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
load_dotenv(REPO_ROOT / '.env')

print('repo root :', REPO_ROOT)
print('notebook  :', NOTEBOOK_DIR)
print('OPENAI_API_KEY :', 'set' if os.environ.get('OPENAI_API_KEY') else 'MISSING')
print('GEMINI_API_KEY :', 'set' if os.environ.get('GEMINI_API_KEY') else 'MISSING')
print('GOOGLE_CLOUD_PROJECT :', os.environ.get('GOOGLE_CLOUD_PROJECT') or '(unset — Imagen disabled)')

## 2. Load the 9 example tiles

`./example/0.png` .. `./example/8.png` — top-left to bottom-right, **index 4 = center**.

In [ ]:
import matplotlib.pyplot as plt

EXAMPLE_DIR = NOTEBOOK_DIR / 'example'

tiles = [Image.open(EXAMPLE_DIR / f'{i}.png').convert('RGBA') for i in range(9)]

assert len({t.size for t in tiles}) == 1, 'tiles are not all the same size'
TILE_SIZE = tiles[0].size[0]
print(f'loaded 9 tiles at {TILE_SIZE}×{TILE_SIZE}')

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for idx in range(9):
    ax = axes[idx // 3][idx % 3]
    ax.imshow(tiles[idx])
    ax.set_title(f'index {idx}')
    ax.axis('off')
fig.suptitle('inputs (index 4 = center / target)')
plt.tight_layout()
plt.show()

## 3. Source loader

`resolve_source(x)` turns any of {folder path, file path, `PIL.Image`, bytes} into a single PIL image ready to send. For folders, the 9 tiles get stitched into a 3×3 grid at `SLOT_SIZE` per cell.

In [ ]:
SLOT_SIZE = 341  # one cell of a 3×3 stitched grid; 3*341 = 1023, close to OpenAI's 1024 size cap


def load_folder_as_grid(folder, slot_size: int = SLOT_SIZE) -> Image.Image:
    """Stitch 9 PNGs from `folder` into a single 3×3 grid image.

    Accepts either naming convention:
      - `0.png` … `8.png` in reading order (top-left → bottom-right), or
      - `c{col}_r{row}.png` for col, row ∈ {-1, 0, 1} (renderer-samples format,
        where row=+1 is the top row).

    Each tile is resized to `slot_size × slot_size` before pasting. Returns an
    RGB image of size `(3*slot_size) × (3*slot_size)`.
    """
    folder = Path(folder)
    if not folder.is_dir():
        raise ValueError(f'not a directory: {folder}')

    cr_files = sorted(folder.glob('c*_r*.png'))
    if cr_files:
        out = Image.new('RGB', (slot_size * 3, slot_size * 3), (0, 0, 0))
        for f in cr_files:
            try:
                col_part, row_part = f.stem.split('_')
                col, row = int(col_part[1:]), int(row_part[1:])
            except (ValueError, IndexError):
                continue
            grid_row = 1 - row  # row=+1 (north) → grid row 0 (top)
            grid_col = col + 1
            if not (0 <= grid_row <= 2 and 0 <= grid_col <= 2):
                continue
            tile = Image.open(f).convert('RGB').resize((slot_size, slot_size), Image.LANCZOS)
            out.paste(tile, (grid_col * slot_size, grid_row * slot_size))
        return out

    # Indexed format: 0.png .. 8.png
    for i in range(9):
        if not (folder / f'{i}.png').exists():
            raise ValueError(f'missing tile: {folder / f"{i}.png"}')
    out = Image.new('RGB', (slot_size * 3, slot_size * 3), (0, 0, 0))
    for i in range(9):
        r, c = divmod(i, 3)
        tile = Image.open(folder / f'{i}.png').convert('RGB').resize(
            (slot_size, slot_size), Image.LANCZOS,
        )
        out.paste(tile, (c * slot_size, r * slot_size))
    return out


def resolve_source(source) -> Image.Image:
    """Coerce a source spec → PIL image (RGB).
      - Path / str: directory → stitched grid; file → loaded as-is
      - PIL.Image: returned (converted to RGB)
      - bytes: decoded
    """
    if isinstance(source, (Path, str)):
        p = Path(source)
        if p.is_dir():
            return load_folder_as_grid(p)
        if p.is_file():
            return Image.open(p).convert('RGB')
        raise ValueError(f'source path does not exist: {p}')
    if isinstance(source, Image.Image):
        return source.convert('RGB')
    if isinstance(source, (bytes, bytearray)):
        return Image.open(io.BytesIO(source)).convert('RGB')
    raise TypeError(f'unsupported source type: {type(source).__name__}')


def to_png_bytes(img: Image.Image) -> bytes:
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return buf.getvalue()


# Quick preview: example folder stitched vs single tile.
preview_grid = load_folder_as_grid(EXAMPLE_DIR)
preview_single = resolve_source(EXAMPLE_DIR / '4.png')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(preview_grid)
axes[0].set_title(f'stitched 3×3 grid · {preview_grid.size[0]}×{preview_grid.size[1]}')
axes[0].axis('off')
axes[1].imshow(preview_single)
axes[1].set_title(f'single tile (idx 4) · {preview_single.size[0]}×{preview_single.size[1]}')
axes[1].axis('off')
plt.tight_layout(); plt.show()

### Available renderer samples

Lists every `sample{i}/` folder under `data/renderer/samples/` (the format `/debug/renderer`'s "render random samples" button produces) and shows a thumbnail of each stitched grid. The `samples` list below lets you reference any of them by index.

In [ ]:
SAMPLES_ROOT = REPO_ROOT / 'data' / 'renderer' / 'samples'


def list_sample_folders(root: Path = SAMPLES_ROOT) -> list[Path]:
    """Return every leaf `sample{i}` folder containing renderer tiles.
    Newest first (sorted by mtime).
    """
    if not root.is_dir():
        return []
    out: list[Path] = []
    for run in root.iterdir():
        if not run.is_dir():
            continue
        for sample in run.iterdir():
            if sample.is_dir() and any(sample.glob('c*_r*.png')):
                out.append(sample)
    out.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return out


samples: list[Path] = list_sample_folders()
print(f'found {len(samples)} sample folder(s) under {SAMPLES_ROOT.relative_to(REPO_ROOT)}/')
for i, s in enumerate(samples):
    print(f'  [{i}] {s.relative_to(REPO_ROOT)}')

if samples:
    cols = min(4, len(samples))
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    flat = (
        [axes] if rows == 1 and cols == 1
        else list(axes) if rows == 1
        else [ax for row in axes for ax in row]
    )
    for idx, ax in enumerate(flat):
        if idx < len(samples):
            grid = load_folder_as_grid(samples[idx], slot_size=120)
            ax.imshow(grid)
            run_name = samples[idx].parent.name
            ax.set_title(f'[{idx}] {run_name}/{samples[idx].name}', fontsize=8)
        ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('no samples yet — run /debug/renderer → "render random samples"')

## 4. Image provider

Single method: `.generate(image_png_bytes, prompt) -> response_png_bytes`. No mask — the provider just runs image-to-image edits over the whole image.

In [ ]:
class ImageProvider(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

    @abstractmethod
    def generate(self, image: bytes, prompt: str) -> bytes: ...


# `input_fidelity` is only accepted by gpt-image-1 / gpt-image-1.5 — gpt-image-2
# rejects the param with HTTP 400, so we silently strip it for that model.
_MODELS_WITH_INPUT_FIDELITY = {'gpt-image-1', 'gpt-image-1.5'}


class OpenAIEdit(ImageProvider):
    """OpenAI image-to-image via /v1/images/edits with **no mask**. The model
    sees the full input image and restyles it according to `prompt`. Clean
    image-to-image — no hybrid-layout games and no mask-inversion confusion.
    """

    def __init__(
        self,
        model_id: str,
        api_key: Optional[str] = None,
        size: str = '1024x1024',
        input_fidelity: Optional[str] = None,
    ):
        from openai import OpenAI

        key = api_key or os.environ.get('OPENAI_API_KEY')
        if not key:
            raise RuntimeError('OPENAI_API_KEY not set in .env')
        self._client = OpenAI(api_key=key)
        self._model_id = model_id
        self._size = size
        if input_fidelity and model_id not in _MODELS_WITH_INPUT_FIDELITY:
            print(
                f'[OpenAIEdit] {model_id} does not accept input_fidelity — dropping it'
            )
            input_fidelity = None
        self._input_fidelity = input_fidelity

    @property
    def name(self) -> str:
        return self._model_id

    def generate(self, image: bytes, prompt: str) -> bytes:
        kwargs: dict = dict(
            model=self._model_id,
            image=('input.png', image, 'image/png'),
            prompt=prompt,
            n=1,
            size=self._size,
        )
        if self._input_fidelity:
            kwargs['input_fidelity'] = self._input_fidelity
        res = self._client.images.edit(**kwargs)
        item = res.data[0]
        if getattr(item, 'b64_json', None):
            return base64.b64decode(item.b64_json)
        if getattr(item, 'url', None):
            import requests
            r = requests.get(item.url, timeout=60)
            r.raise_for_status()
            return r.content
        raise RuntimeError(f'no image in {self._model_id} response')


GPT_IMAGE_1 = OpenAIEdit('gpt-image-1')
GPT_IMAGE_15 = OpenAIEdit('gpt-image-1.5')
GPT_IMAGE_2 = OpenAIEdit('gpt-image-2')
GPT_IMAGE_1_HIGH = OpenAIEdit('gpt-image-1', input_fidelity='high')
GPT_IMAGE_15_HIGH = OpenAIEdit('gpt-image-1.5', input_fidelity='high')
# gpt-image-2 doesn't accept input_fidelity; this alias resolves to plain GPT_IMAGE_2.
GPT_IMAGE_2_HIGH = GPT_IMAGE_2

print('providers ready:', [p.name for p in (GPT_IMAGE_1, GPT_IMAGE_15, GPT_IMAGE_2)])

## 5. `generate_from(source, provider)`

Resolves the source (file/folder/PIL/bytes) into a single image, sends it to the provider, shows source + output side by side, and returns both.

In [ ]:
from datetime import datetime

DEFAULT_PROMPT = (
    'Convert this aerial isometric city render into a 16-bit isometric pixel-art image '
    'in the style of SimCity 2000, RollerCoaster Tycoon 2, and Theme Hospital. Late-1990s '
    'simulation game aesthetic: limited saturated palette, crisp aliased pixel edges, '
    'simple flat shading with a single top-left light direction. Use the satellite content '
    'visible in the input as the strict source — every building, road, intersection, tree, '
    'water surface, and open lot in your output must correspond 1:1 to the input. Do not '
    'add, remove, relocate, resize, or invent anything. If unsure about a detail, prefer '
    'the input over your own priors. Treat low-poly artifacts as cues about real-world '
    'content, not features to copy: blocky tree shapes are trees (round pixel-art crowns), '
    'shimmering surfaces are water (flat color + 2-pixel checkerboard), stretched facades '
    'are buildings (clean rectangular pixel-art walls).'
)

GENERATED_ROOT = REPO_ROOT / 'data' / 'generated' / 'notebook'


def _resolve_save_dir(source, provider_name: str) -> Path:
    """Pick where to dump source.png / output.png / prompt.txt for a run."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    safe_provider = provider_name.replace('/', '_').replace(' ', '_')
    if isinstance(source, (str, Path)):
        p = Path(source)
        if p.is_dir():
            # Live alongside the source folder so generations stay grouped with their inputs.
            return p / 'generated' / f'{ts}_{safe_provider}'
        if p.is_file():
            return p.parent / 'generated' / f'{ts}_{safe_provider}_{p.stem}'
    # In-memory source (PIL/bytes) → central notebook bucket.
    return GENERATED_ROOT / f'{ts}_{safe_provider}'


def generate_from(
    source,
    provider: ImageProvider,
    prompt: str = DEFAULT_PROMPT,
    show: bool = True,
    save: bool = True,
) -> dict:
    """Restyle `source` via `provider` and (by default) persist the result.

    `source` is anything `resolve_source` accepts — folder path, file path,
    `PIL.Image`, or raw PNG bytes. When `save=True`, writes three files in
    a fresh timestamped folder:
      - `source.png`   the exact bytes sent to the model
      - `output.png`   the model's response
      - `prompt.txt`   the prompt used
    Save location is alongside the source folder/file, or under
    `data/generated/notebook/` for in-memory sources.
    """
    src_img = resolve_source(source)
    src_bytes = to_png_bytes(src_img)

    raw = provider.generate(src_bytes, prompt)
    raw_img = Image.open(io.BytesIO(raw))

    save_dir: Optional[Path] = None
    if save:
        save_dir = _resolve_save_dir(source, provider.name)
        save_dir.mkdir(parents=True, exist_ok=True)
        (save_dir / 'source.png').write_bytes(src_bytes)
        (save_dir / 'output.png').write_bytes(raw)
        (save_dir / 'prompt.txt').write_text(prompt)
        try:
            rel = save_dir.relative_to(REPO_ROOT)
            print(f'saved → {rel}')
        except ValueError:
            print(f'saved → {save_dir}')

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(src_img)
        axes[0].set_title(f'source  {src_img.size[0]}×{src_img.size[1]}')
        axes[0].axis('off')
        axes[1].imshow(raw_img)
        axes[1].set_title(f'output  [{provider.name}]  {raw_img.size[0]}×{raw_img.size[1]}')
        axes[1].axis('off')
        plt.tight_layout(); plt.show()

    return {
        'source_image': src_img,
        'source_bytes': src_bytes,
        'raw': raw,
        'raw_image': raw_img,
        'save_dir': save_dir,
    }

## 6. Test runs

Pick a source and a provider. Each call hits the API.

In [ ]:
# Single tile — center of the example.
out_single = generate_from(EXAMPLE_DIR / '4.png', GPT_IMAGE_2)

In [ ]:
# Whole example folder → stitched 3×3 grid → restyled.
out_grid = generate_from(EXAMPLE_DIR, GPT_IMAGE_2)

In [ ]:
# Same source via gpt-image-1.5 — compare against gpt-image-2 above.
out_15 = generate_from(EXAMPLE_DIR, GPT_IMAGE_15)

In [ ]:
# gpt-image-2 with input_fidelity='high' — strongest preservation knob OpenAI exposes.
out_high = generate_from(EXAMPLE_DIR, GPT_IMAGE_2_HIGH)

In [ ]:
# Restyle a renderer sample (stitched 3×3) by index in the `samples` list above.
if samples:
    out_renderer = generate_from(samples[0], GPT_IMAGE_15)
else:
    print('no samples available — re-run the listing cell above after rendering some')

In [ ]:
if samples:
    for sample in samples[0:3]:
        out_single = generate_from(sample / 'c1_r1.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c0_r1.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c1_r0.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c0_r0.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c-1_r1.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c-1_r0.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c-1_r-1.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c0_r-1.png', GPT_IMAGE_15)
        out_single = generate_from(sample / 'c1_r-1.png', GPT_IMAGE_15)        
        

In [ ]:
if samples:
    for sample in samples[0:3]:
        out_single = generate_from(sample / 'c1_r1.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c0_r1.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c1_r0.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c0_r0.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c-1_r1.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c-1_r0.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c-1_r-1.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c0_r-1.png', GPT_IMAGE_15_HIGH)
        out_single = generate_from(sample / 'c1_r-1.png', GPT_IMAGE_15_HIGH)    

In [ ]:
if samples:
    for sample in samples[0:3]:
        out_single = generate_from(sample / 'c1_r1.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c0_r1.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c1_r0.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c0_r0.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c-1_r1.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c-1_r0.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c-1_r-1.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c0_r-1.png', GPT_IMAGE_2)
        out_single = generate_from(sample / 'c1_r-1.png', GPT_IMAGE_2)        
        
